In [ ]:
# @title 0) Colab 커널 bootstrap: 저장소 clone/update + 최소 설치
import os
import subprocess
import sys
from pathlib import Path

print("[bootstrap] start", flush=True)

REPO_URL = "https://github.com/JeonDongJun/mindscopex_analysis"
TARGET = Path("/content/colab")
MARKER = Path("src") / "mindscopex_analysis" / "__init__.py"


def run(cmd, cwd=None, timeout=300, required=True):
    print("+", " ".join(map(str, cmd)), flush=True)
    try:
        subprocess.run(
            cmd,
            cwd=str(cwd) if cwd else None,
            timeout=timeout,
            check=True,
        )
    except Exception as exc:
        print(f"[bootstrap] command failed: {exc}", flush=True)
        if required:
            raise


if Path("/content").exists():
    if (TARGET / ".git").exists():
        run(["git", "fetch", "origin", "main"], cwd=TARGET, timeout=90, required=False)
        run(["git", "checkout", "main"], cwd=TARGET, timeout=60, required=False)
        run(["git", "pull", "--ff-only", "origin", "main"], cwd=TARGET, timeout=90, required=False)
    elif not TARGET.exists():
        run(["git", "clone", "--depth", "1", REPO_URL, str(TARGET)], timeout=180)
    elif not (TARGET / MARKER).is_file():
        alt = Path("/content/mindscopex_analysis")
        if (alt / ".git").exists():
            run(["git", "fetch", "origin", "main"], cwd=alt, timeout=90, required=False)
            run(["git", "checkout", "main"], cwd=alt, timeout=60, required=False)
            run(["git", "pull", "--ff-only", "origin", "main"], cwd=alt, timeout=90, required=False)
        elif not alt.exists():
            run(["git", "clone", "--depth", "1", REPO_URL, str(alt)], timeout=180)
        TARGET = alt
    if not (TARGET / MARKER).is_file():
        raise FileNotFoundError(f"저장소 marker를 찾지 못했습니다: {TARGET / MARKER}")
    os.chdir(TARGET)
    os.environ["MINDSCOPEX_ROOT"] = str(TARGET)
    print("cwd =", Path.cwd(), flush=True)
    print("MINDSCOPEX_ROOT =", os.environ["MINDSCOPEX_ROOT"], flush=True)
    run([sys.executable, "-m", "pip", "install", "-e", "."], timeout=600)
    print("[bootstrap] done", flush=True)
else:
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / MARKER).is_file():
            os.environ["MINDSCOPEX_ROOT"] = str(base)
            print("로컬 저장소에서 실행 중입니다:", base, flush=True)
            break
    else:
        print("저장소 루트를 찾지 못했습니다. Colab 커널이면 이 셀을 맨 먼저 다시 실행하세요.", flush=True)


# CRT bat-and-ball lure feature MVP

목표는 전체 RQ1을 돌리기 전에, bat-and-ball CRT에서 아래 패턴을 보이는 SAE feature가 실제로 있는지 최소 실험으로 확인하는 것입니다.

| 입력/상황 | 기대 activation |
|---|---|
| bat-and-ball 문제에서 답을 내기 직전 | 높음 |
| 모델이 `10 cents`라고 틀리게 답하는 run | 매우 높음 |
| 모델이 `5 cents`라고 맞히는 run의 초기 단계 | 잠깐 높음 |
| 모델이 식을 세우고 검산하는 후반 단계 | 낮아짐 |
| 단순히 `10 cents`가 나오는 일반 문장 | 낮거나 중간 |
| 함정이 없는 돈 계산 문제 | 낮음 |

이 노트북은 실제 generation을 먼저 신뢰하지 않고, 위 상황을 forced transcript / probe context로 만들어 Qwen3-Base residual stream을 Qwen-Scope SAE에 투영합니다. 발견은 일부 paraphrase에서 하고, 검증은 남겨둔 paraphrase에서 봅니다.


## 1. Import와 설정


In [ ]:
import gc
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

_REPO_MARK = Path("src") / "mindscopex_analysis" / "__init__.py"


def _find_repo_root() -> Path:
    env = os.environ.get("MINDSCOPEX_ROOT", "").strip()
    if env:
        root = Path(env).expanduser().resolve()
        if (root / _REPO_MARK).is_file():
            return root
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / _REPO_MARK).is_file():
            return base
    raise FileNotFoundError("src/mindscopex_analysis 를 찾지 못했습니다.")


ROOT = _find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mindscopex_analysis.notebook_utils import dtype_from_str
from mindscopex_analysis.qwen_scope import (
    capture_residuals,
    get_transformer_layers,
    load_qwen_scope_sae,
    summarize_qwen_scope_features,
)

pio.renderers.default = "plotly_mimetype"
set_seed(42)
print("ROOT =", ROOT)
print("torch =", torch.__version__, "cuda =", torch.cuda.is_available())


In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B-Base"
SAE_REPO = "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = dtype_from_str("bfloat16" if DEVICE == "cuda" else "float32")
LAYERS = [6, 14, 21, 27]
SAE_TOP_K = 50
BATCH_SIZE = 64
MAX_LENGTH = 1024
TOKEN_POSITION = "last"
DISCOVERY_TOP_N = 40
REPORT_TOP_N = 20

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
OUT_DIR = ROOT / "outputs" / "crt_lure_feature_mvp" / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"model={MODEL_ID}")
print(f"sae={SAE_REPO}")
print(f"device={DEVICE} dtype={DTYPE} layers={LAYERS}")
print("out:", OUT_DIR)


## 2. Probe contexts


In [ ]:
BAT_BALL = (
    "A bat and a ball cost $1.10 in total. "
    "The bat costs $1.00 more than the ball. "
    "How much does the ball cost?"
)

PROBES = [
    # 답을 내기 직전: lure가 올라오는지
    {
        "id": "pre_answer_0",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "discovery",
        "text": BAT_BALL + " Think briefly, then answer. The ball costs",
    },
    {
        "id": "pre_answer_1",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "discovery",
        "text": BAT_BALL + " The quick answer is",
    },
    {
        "id": "pre_answer_2",
        "condition": "lure_pre_answer",
        "expected_level": "high",
        "split": "validation",
        "text": BAT_BALL + " Final answer:",
    },
    # 오답 run: 직관 오답 10 cents를 말하는 transcript
    {
        "id": "wrong_10_0",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "discovery",
        "text": BAT_BALL + " The ball costs 10 cents. Final answer: 10 cents.",
    },
    {
        "id": "wrong_10_1",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "discovery",
        "text": BAT_BALL + " It seems straightforward: the ball is 10 cents.",
    },
    {
        "id": "wrong_10_2",
        "condition": "wrong_10_answer",
        "expected_level": "very_high",
        "split": "validation",
        "text": BAT_BALL + " My answer is 10 cents.",
    },
    # 정답 run 초기: 처음에는 10 cents lure가 떠오르지만 아직 수정 전
    {
        "id": "correct_initial_0",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "discovery",
        "text": BAT_BALL + " At first glance, the ball seems to cost 10 cents, but",
    },
    {
        "id": "correct_initial_1",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "discovery",
        "text": BAT_BALL + " A tempting answer is 10 cents. However, checking it,",
    },
    {
        "id": "correct_initial_2",
        "condition": "correct_initial_lure",
        "expected_level": "transient_high",
        "split": "validation",
        "text": BAT_BALL + " The intuitive guess is 10 cents, but that would make",
    },
    # 정답 run 후반: 식과 검산 후 5 cents로 안정화
    {
        "id": "late_check_0",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "discovery",
        "text": BAT_BALL + " Let x be the ball. Then x + (x + 100 cents) = 110 cents, so 2x = 10 and x = 5. Final answer: 5 cents.",
    },
    {
        "id": "late_check_1",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "discovery",
        "text": BAT_BALL + " Check: if the ball is 5 cents, the bat is 105 cents, total 110 cents. Therefore the ball is 5 cents.",
    },
    {
        "id": "late_check_2",
        "condition": "correct_late_check",
        "expected_level": "low_after_check",
        "split": "validation",
        "text": BAT_BALL + " Solving carefully gives ball = 5 cents and bat = 105 cents. Answer: 5 cents.",
    },
    # 단순 10 cents: 숫자/토큰 자체 feature를 배제하기 위한 control
    {
        "id": "plain_10_0",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "discovery",
        "text": "The sticker on the pencil says 10 cents. There is no puzzle; the listed price is 10 cents.",
    },
    {
        "id": "plain_10_1",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "discovery",
        "text": "A parking meter displays 10 cents remaining. This sentence simply mentions 10 cents.",
    },
    {
        "id": "plain_10_2",
        "condition": "plain_10_sentence",
        "expected_level": "low_or_mid",
        "split": "validation",
        "text": "The donation jar contains a coin worth 10 cents. Nothing is being solved.",
    },
    # 함정 없는 돈 계산
    {
        "id": "money_control_0",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "discovery",
        "text": "A bat costs $1.05 and a ball costs $0.05. Together they cost $1.10. How much does the ball cost? Answer: 5 cents.",
    },
    {
        "id": "money_control_1",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "discovery",
        "text": "A book costs $2.15 and a toy costs $0.15. The total is $2.30. How much does the toy cost? Answer: 15 cents.",
    },
    {
        "id": "money_control_2",
        "condition": "no_lure_money_control",
        "expected_level": "low",
        "split": "validation",
        "text": "A pen costs 95 cents and an eraser costs 5 cents. What is the eraser's price? Answer: 5 cents.",
    },
]

probe_df = pd.DataFrame(PROBES)
display(probe_df[["id", "condition", "expected_level", "split", "text"]])
print(probe_df.groupby(["condition", "split"]).size().unstack(fill_value=0))


## 3. Residual capture와 Qwen-Scope projection


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    trust_remote_code=True,
).to(DEVICE).eval()

residual_by_probe = {}
for probe in tqdm(PROBES, desc="capture residuals"):
    residual_by_probe[probe["id"]] = capture_residuals(
        model,
        tokenizer,
        [probe["text"]],
        LAYERS,
        device=DEVICE,
        max_length=MAX_LENGTH,
        token_position=TOKEN_POSITION,
    )

del model, tokenizer
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("captured", len(residual_by_probe), "probe contexts")


In [ ]:
feature_matrices = {}
feature_rows = []

for layer in tqdm(LAYERS, desc="SAE projection"):
    sae = load_qwen_scope_sae(
        SAE_REPO,
        layer,
        device=DEVICE,
        dtype=DTYPE,
        top_k=SAE_TOP_K,
    )
    rows = []
    ids = []
    for probe in PROBES:
        summary = summarize_qwen_scope_features(
            residual_by_probe[probe["id"]][layer],
            sae,
            batch_size=BATCH_SIZE,
        )
        mean_vec = summary["mean"].numpy()
        max_vec = summary["max"].numpy()
        rate_vec = summary["activation_rate"].numpy()
        rows.append(mean_vec)
        ids.append(probe["id"])
        top_idx = np.argsort(mean_vec)[::-1][:10]
        for rank, feature in enumerate(top_idx, start=1):
            feature_rows.append({
                "layer": layer,
                "probe_id": probe["id"],
                "condition": probe["condition"],
                "split": probe["split"],
                "feature": int(feature),
                "rank_in_probe": rank,
                "mean_activation": float(mean_vec[feature]),
                "max_activation": float(max_vec[feature]),
                "activation_rate": float(rate_vec[feature]),
            })
    feature_matrices[layer] = {"probe_ids": ids, "matrix": np.stack(rows)}
    del sae
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

feature_rows_df = pd.DataFrame(feature_rows)
feature_rows_df.to_csv(OUT_DIR / "probe_top_features.csv", index=False, encoding="utf-8-sig")
display(feature_rows_df.head(20))


## 4. Discovery: lure 후보 feature 찾기


In [ ]:
HIGH_CONDITIONS = {"lure_pre_answer", "wrong_10_answer", "correct_initial_lure"}
LOW_CONDITIONS = {"correct_late_check", "plain_10_sentence", "no_lure_money_control"}

probe_meta = {p["id"]: p for p in PROBES}
candidate_rows = []

for layer, bundle in feature_matrices.items():
    ids = bundle["probe_ids"]
    X = bundle["matrix"]
    is_discovery = np.array([probe_meta[i]["split"] == "discovery" for i in ids])
    conds = np.array([probe_meta[i]["condition"] for i in ids])

    high = is_discovery & np.isin(conds, list(HIGH_CONDITIONS))
    low = is_discovery & np.isin(conds, list(LOW_CONDITIONS))
    wrong = is_discovery & (conds == "wrong_10_answer")
    plain10 = is_discovery & (conds == "plain_10_sentence")
    early = is_discovery & (conds == "correct_initial_lure")
    late = is_discovery & (conds == "correct_late_check")
    money = is_discovery & (conds == "no_lure_money_control")

    high_mean = X[high].mean(axis=0)
    low_mean = X[low].mean(axis=0)
    pooled = X[is_discovery].std(axis=0) + 1e-8
    discovery_effect = (high_mean - low_mean) / pooled

    wrong_minus_plain10 = X[wrong].mean(axis=0) - X[plain10].mean(axis=0)
    early_minus_late = X[early].mean(axis=0) - X[late].mean(axis=0)
    pre_minus_money = X[is_discovery & (conds == "lure_pre_answer")].mean(axis=0) - X[money].mean(axis=0)

    # 숫자 "10 cents" 자체가 아니라 CRT-lure 문맥에 특이적인 feature를 선호한다.
    score = discovery_effect + 0.5 * np.sign(wrong_minus_plain10) + 0.5 * np.sign(early_minus_late)
    top = np.argsort(score)[::-1][:DISCOVERY_TOP_N]

    for rank, feature in enumerate(top, start=1):
        candidate_rows.append({
            "layer": layer,
            "rank": rank,
            "feature": int(feature),
            "discovery_score": float(score[feature]),
            "high_mean": float(high_mean[feature]),
            "low_mean": float(low_mean[feature]),
            "discovery_effect": float(discovery_effect[feature]),
            "wrong_minus_plain10": float(wrong_minus_plain10[feature]),
            "early_minus_late": float(early_minus_late[feature]),
            "pre_minus_money": float(pre_minus_money[feature]),
        })

candidates = (
    pd.DataFrame(candidate_rows)
    .sort_values("discovery_score", ascending=False)
    .reset_index(drop=True)
)
candidates.to_csv(OUT_DIR / "candidate_lure_features_discovery.csv", index=False, encoding="utf-8-sig")
display(candidates.head(REPORT_TOP_N))


## 5. Validation: 남겨둔 paraphrase에서 기대 패턴 확인


In [ ]:
validation_rows = []

for _, cand in candidates.head(DISCOVERY_TOP_N).iterrows():
    layer = int(cand["layer"])
    feature = int(cand["feature"])
    bundle = feature_matrices[layer]
    ids = bundle["probe_ids"]
    X = bundle["matrix"]
    rows = []
    for i, pid in enumerate(ids):
        meta = probe_meta[pid]
        rows.append({
            "layer": layer,
            "feature": feature,
            "probe_id": pid,
            "condition": meta["condition"],
            "split": meta["split"],
            "activation": float(X[i, feature]),
        })
    df_feat = pd.DataFrame(rows)
    val = df_feat[df_feat["split"] == "validation"]
    means = val.groupby("condition")["activation"].mean().to_dict()

    checks = {
        "wrong_gt_plain10": means.get("wrong_10_answer", 0.0) > means.get("plain_10_sentence", 0.0),
        "wrong_gt_money": means.get("wrong_10_answer", 0.0) > means.get("no_lure_money_control", 0.0),
        "pre_gt_money": means.get("lure_pre_answer", 0.0) > means.get("no_lure_money_control", 0.0),
        "early_gt_late": means.get("correct_initial_lure", 0.0) > means.get("correct_late_check", 0.0),
        "late_le_wrong": means.get("correct_late_check", 0.0) <= means.get("wrong_10_answer", 0.0),
    }
    validation_rows.append({
        "layer": layer,
        "feature": feature,
        "discovery_rank": int(cand["rank"]),
        "discovery_score": float(cand["discovery_score"]),
        **{f"val_{k}": float(v) for k, v in means.items()},
        "pattern_score": int(sum(checks.values())),
        **checks,
    })

validation = pd.DataFrame(validation_rows).sort_values(
    ["pattern_score", "discovery_score"],
    ascending=[False, False],
)
validation.to_csv(OUT_DIR / "validation_pattern_scores.csv", index=False, encoding="utf-8-sig")
display(validation.head(REPORT_TOP_N))


In [ ]:
if validation.empty:
    raise ValueError("validation 결과가 비었습니다.")

PICK_LAYER = int(validation.iloc[0]["layer"])
PICK_FEATURE = int(validation.iloc[0]["feature"])
print("selected feature:", f"L{PICK_LAYER}/F{PICK_FEATURE}")

bundle = feature_matrices[PICK_LAYER]
plot_rows = []
for i, pid in enumerate(bundle["probe_ids"]):
    meta = probe_meta[pid]
    plot_rows.append({
        "probe_id": pid,
        "condition": meta["condition"],
        "split": meta["split"],
        "expected_level": meta["expected_level"],
        "activation": float(bundle["matrix"][i, PICK_FEATURE]),
    })
plot_df = pd.DataFrame(plot_rows)
order = [
    "lure_pre_answer",
    "wrong_10_answer",
    "correct_initial_lure",
    "correct_late_check",
    "plain_10_sentence",
    "no_lure_money_control",
]
plot_df["condition"] = pd.Categorical(plot_df["condition"], categories=order, ordered=True)
plot_df = plot_df.sort_values(["condition", "split", "probe_id"])
display(plot_df)

fig = go.Figure()
for split, sdf in plot_df.groupby("split", observed=False):
    fig.add_trace(go.Bar(
        x=[f"{r.condition}<br>{r.probe_id}" for r in sdf.itertuples()],
        y=sdf["activation"],
        name=split,
    ))
fig.update_layout(
    title=f"CRT lure MVP selected feature activation: L{PICK_LAYER}/F{PICK_FEATURE}",
    xaxis_title="condition / probe",
    yaxis_title="SAE mean activation at last token",
    template="plotly_white",
    height=520,
)
fig.show()

heat = plot_df.pivot_table(
    index="condition",
    columns="probe_id",
    values="activation",
    observed=False,
).reindex(order).fillna(0.0)
fig2 = go.Figure(data=go.Heatmap(
    z=heat.values,
    x=heat.columns.tolist(),
    y=heat.index.astype(str).tolist(),
    colorscale="Viridis",
))
fig2.update_layout(
    title=f"Heatmap for selected feature L{PICK_LAYER}/F{PICK_FEATURE}",
    template="plotly_white",
    height=420,
)
fig2.show()


## 7. H100용 Qwen/Qwen-Scope 모델군 비교

Qwen-Scope collection 기준으로 현재 feature projection에 바로 쓸 수 있는 최신 축은 Qwen3와 Qwen3.5입니다. Qwen3.6은 더 최신 모델군이지만, 이 노트북이 쓰는 Qwen-Scope SAE repo가 아직 collection에 보이지 않으므로 여기서는 제외합니다.

H100 한 장에서는 모델을 동시에 올리지 않고 **순차 로드 → projection → unload** 방식으로 실행합니다. 기본 suite는 H100에서 현실적인 검증 범위인 `qwen3_1p7b`, `qwen35_2b`, `qwen3_8b`, `qwen35_9b`입니다. 더 무겁게 보고 싶으면 `SUITE_NAME = "h100_extended"`로 바꾸세요.


In [ ]:
MODEL_SUITES = {
    "h100_recommended": [
        {
            "tag": "qwen3_1p7b_base",
            "model_id": "Qwen/Qwen3-1.7B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_2b_base",
            "model_id": "Qwen/Qwen3.5-2B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_8b_base",
            "model_id": "Qwen/Qwen3-8B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_9b_base",
            "model_id": "Qwen/Qwen3.5-9B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-9B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
    ],
    "h100_extended": [
        {
            "tag": "qwen3_1p7b_base",
            "model_id": "Qwen/Qwen3-1.7B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-1.7B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_2b_base",
            "model_id": "Qwen/Qwen3.5-2B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-2B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_8b_base",
            "model_id": "Qwen/Qwen3-8B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-8B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_9b_base",
            "model_id": "Qwen/Qwen3.5-9B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-9B-Base-W64K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen3_30b_a3b_base",
            "model_id": "Qwen/Qwen3-30B-A3B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3-30B-A3B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_35b_a3b_base",
            "model_id": "Qwen/Qwen3.5-35B-A3B-Base",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-35B-A3B-Base-W32K-L0_50",
            "layer_policy": "quartiles",
        },
        {
            "tag": "qwen35_27b",
            "model_id": "Qwen/Qwen3.5-27B",
            "sae_repo": "Qwen/SAE-Res-Qwen3.5-27B-W80K-L0_50",
            "layer_policy": "quartiles",
        },
    ],
}

pd.DataFrame(
    [
        {"suite": suite, **spec}
        for suite, specs in MODEL_SUITES.items()
        for spec in specs
    ]
).drop_duplicates(["suite", "tag"]).pipe(display)


In [ ]:
def choose_probe_layers(model, policy="quartiles", explicit_layers=None):
    if explicit_layers:
        return [int(x) for x in explicit_layers]
    blocks = get_transformer_layers(model)
    n_layers = len(blocks)
    if policy == "quartiles":
        raw = [round(n_layers * q) for q in (0.25, 0.50, 0.75, 0.95)]
    elif policy == "middle_last":
        raw = [n_layers // 2, n_layers - 1]
    else:
        raise ValueError(f"unknown layer policy: {policy}")
    return sorted({min(max(int(x), 0), n_layers - 1) for x in raw})


def load_suite_model(model_id: str):
    kwargs = {
        "torch_dtype": DTYPE,
        "trust_remote_code": True,
        "low_cpu_mem_usage": True,
    }
    if DEVICE == "cuda":
        kwargs["device_map"] = "auto"
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs).eval()
    if DEVICE != "cuda":
        model = model.to(DEVICE)
    return model


def score_lure_candidates(local_feature_matrices, local_layers):
    rows = []
    probe_meta = {p["id"]: p for p in PROBES}
    high_conditions = {"lure_pre_answer", "wrong_10_answer", "correct_initial_lure"}
    low_conditions = {"correct_late_check", "plain_10_sentence", "no_lure_money_control"}

    for layer in local_layers:
        bundle = local_feature_matrices[layer]
        ids = bundle["probe_ids"]
        X = bundle["matrix"]
        is_discovery = np.array([probe_meta[i]["split"] == "discovery" for i in ids])
        conds = np.array([probe_meta[i]["condition"] for i in ids])

        high = is_discovery & np.isin(conds, list(high_conditions))
        low = is_discovery & np.isin(conds, list(low_conditions))
        wrong = is_discovery & (conds == "wrong_10_answer")
        plain10 = is_discovery & (conds == "plain_10_sentence")
        early = is_discovery & (conds == "correct_initial_lure")
        late = is_discovery & (conds == "correct_late_check")
        money = is_discovery & (conds == "no_lure_money_control")

        high_mean = X[high].mean(axis=0)
        low_mean = X[low].mean(axis=0)
        pooled = X[is_discovery].std(axis=0) + 1e-8
        discovery_effect = (high_mean - low_mean) / pooled
        wrong_minus_plain10 = X[wrong].mean(axis=0) - X[plain10].mean(axis=0)
        early_minus_late = X[early].mean(axis=0) - X[late].mean(axis=0)
        pre_minus_money = (
            X[is_discovery & (conds == "lure_pre_answer")].mean(axis=0)
            - X[money].mean(axis=0)
        )
        score = discovery_effect + 0.5 * np.sign(wrong_minus_plain10) + 0.5 * np.sign(early_minus_late)
        top = np.argsort(score)[::-1][:DISCOVERY_TOP_N]
        for rank, feature in enumerate(top, start=1):
            rows.append({
                "layer": int(layer),
                "rank": rank,
                "feature": int(feature),
                "discovery_score": float(score[feature]),
                "high_mean": float(high_mean[feature]),
                "low_mean": float(low_mean[feature]),
                "discovery_effect": float(discovery_effect[feature]),
                "wrong_minus_plain10": float(wrong_minus_plain10[feature]),
                "early_minus_late": float(early_minus_late[feature]),
                "pre_minus_money": float(pre_minus_money[feature]),
            })
    return (
        pd.DataFrame(rows)
        .sort_values("discovery_score", ascending=False)
        .reset_index(drop=True)
    )


def validate_lure_candidates(local_candidates, local_feature_matrices):
    probe_meta = {p["id"]: p for p in PROBES}
    rows = []
    for _, cand in local_candidates.head(DISCOVERY_TOP_N).iterrows():
        layer = int(cand["layer"])
        feature = int(cand["feature"])
        bundle = local_feature_matrices[layer]
        ids = bundle["probe_ids"]
        X = bundle["matrix"]
        feat_rows = []
        for i, pid in enumerate(ids):
            meta = probe_meta[pid]
            feat_rows.append({
                "probe_id": pid,
                "condition": meta["condition"],
                "split": meta["split"],
                "activation": float(X[i, feature]),
            })
        df_feat = pd.DataFrame(feat_rows)
        val = df_feat[df_feat["split"] == "validation"]
        means = val.groupby("condition")["activation"].mean().to_dict()
        checks = {
            "wrong_gt_plain10": means.get("wrong_10_answer", 0.0) > means.get("plain_10_sentence", 0.0),
            "wrong_gt_money": means.get("wrong_10_answer", 0.0) > means.get("no_lure_money_control", 0.0),
            "pre_gt_money": means.get("lure_pre_answer", 0.0) > means.get("no_lure_money_control", 0.0),
            "early_gt_late": means.get("correct_initial_lure", 0.0) > means.get("correct_late_check", 0.0),
            "late_le_wrong": means.get("correct_late_check", 0.0) <= means.get("wrong_10_answer", 0.0),
        }
        rows.append({
            "layer": layer,
            "feature": feature,
            "discovery_rank": int(cand["rank"]),
            "discovery_score": float(cand["discovery_score"]),
            **{f"val_{k}": float(v) for k, v in means.items()},
            "pattern_score": int(sum(checks.values())),
            **checks,
        })
    return pd.DataFrame(rows).sort_values(
        ["pattern_score", "discovery_score"],
        ascending=[False, False],
    )


In [ ]:
def run_suite_model(spec: dict) -> dict:
    tag = spec["tag"]
    model_id = spec["model_id"]
    sae_repo = spec["sae_repo"]
    model_out = OUT_DIR / "model_suite" / tag
    model_out.mkdir(parents=True, exist_ok=True)

    print(f"\n===== {tag} =====")
    print("model:", model_id)
    print("sae:", sae_repo)

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = load_suite_model(model_id)
    layers = choose_probe_layers(
        model,
        policy=spec.get("layer_policy", "quartiles"),
        explicit_layers=spec.get("layers"),
    )
    print("layers:", layers)

    residual_by_probe = {}
    try:
        for probe in tqdm(PROBES, desc=f"{tag}: capture"):
            residual_by_probe[probe["id"]] = capture_residuals(
                model,
                tokenizer,
                [probe["text"]],
                layers,
                device=DEVICE,
                max_length=MAX_LENGTH,
                token_position=TOKEN_POSITION,
            )
    finally:
        del model, tokenizer
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    local_feature_matrices = {}
    for layer in tqdm(layers, desc=f"{tag}: SAE projection"):
        sae = load_qwen_scope_sae(
            sae_repo,
            layer,
            device=DEVICE,
            dtype=DTYPE,
            top_k=SAE_TOP_K,
        )
        rows = []
        ids = []
        for probe in PROBES:
            summary = summarize_qwen_scope_features(
                residual_by_probe[probe["id"]][layer],
                sae,
                batch_size=BATCH_SIZE,
            )
            rows.append(summary["mean"].numpy())
            ids.append(probe["id"])
        local_feature_matrices[layer] = {"probe_ids": ids, "matrix": np.stack(rows)}
        del sae
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    del residual_by_probe
    gc.collect()

    local_candidates = score_lure_candidates(local_feature_matrices, layers)
    local_validation = validate_lure_candidates(local_candidates, local_feature_matrices)
    local_candidates.to_csv(model_out / "candidate_lure_features.csv", index=False, encoding="utf-8-sig")
    local_validation.to_csv(model_out / "validation_pattern_scores.csv", index=False, encoding="utf-8-sig")

    best = local_validation.iloc[0].to_dict()
    best.update({
        "tag": tag,
        "model_id": model_id,
        "sae_repo": sae_repo,
        "layers": ",".join(map(str, layers)),
        "output_dir": str(model_out),
    })
    print("best:", {k: best[k] for k in ["tag", "layer", "feature", "pattern_score", "discovery_score"]})
    return best


In [ ]:
# H100에서 실행할 때 True로 바꾸세요.
RUN_MODEL_SUITE = False
SUITE_NAME = "h100_recommended"  # "h100_recommended" | "h100_extended"
MAX_MODELS = None  # 예: 2 로 두면 앞의 2개만 smoke test

suite_results = []
if RUN_MODEL_SUITE:
    specs = MODEL_SUITES[SUITE_NAME]
    if MAX_MODELS is not None:
        specs = specs[: int(MAX_MODELS)]
    for spec in specs:
        try:
            suite_results.append(run_suite_model(spec))
        except Exception as exc:
            print(f"[SKIP/FAIL] {spec['tag']}: {type(exc).__name__}: {exc}")
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    suite_df = pd.DataFrame(suite_results)
    suite_df.to_csv(OUT_DIR / f"{SUITE_NAME}_summary.csv", index=False, encoding="utf-8-sig")
    display(suite_df.sort_values(["pattern_score", "discovery_score"], ascending=[False, False]))
else:
    print("RUN_MODEL_SUITE=False 입니다. H100에서 모델군 비교를 돌릴 때 True로 바꾸세요.")


## 8. 해석 기준

MVP에서 가장 먼저 볼 것은 `pattern_score`와 선택 feature의 막대그래프입니다.

- 좋은 후보: `wrong_10_answer`, `lure_pre_answer`, `correct_initial_lure`가 높고 `correct_late_check`, `plain_10_sentence`, `no_lure_money_control`이 낮습니다.
- 애매한 후보: `plain_10_sentence`도 같이 높으면 “10 cents 토큰/가격 문맥” feature일 가능성이 큽니다.
- 실패한 후보: `no_lure_money_control`이 높으면 CRT lure가 아니라 일반 돈 계산 feature일 수 있습니다.

이 MVP가 통과하면 다음 단계는 실제 Qwen3 `think_off`/`think_on` generation run을 수집하고, wrong/correct run의 token trajectory에 같은 feature가 재현되는지 보는 것입니다.
